## **1 - MỤC TIÊU**

Nhiệm vụ trọng tâm của tuần này là tùy chỉnh lại trình nạp dữ liệu (data loader) để mô hình UniFashion có thể đọc được tập dữ liệu FashionIQ mà nhóm đã tiền xử lý.

Trình nạp gốc của UniFashion gặp hạn chế vì bị gán cứng vào định dạng JSON nội bộ và đuôi ảnh .png. Do đó, luồng nạp được thiết kế lại để hệ thống tương thích trực tiếp với file triplet định dạng CSV và ảnh .jpg nằm ở thư mục ngoài.

Notebook tuần này giải quyết khâu xử lý đầu vào từ việc đọc và ánh xạ chính xác các trường thông tin (candidate, modifier, target, category), đóng gói thành tập dữ liệu chuẩn, cho đến việc chạy kiểm thử qua PyTorch DataLoader trên cả mẫu đơn (sample) lẫn nạp theo lô (batch). Quá trình huấn luyện tinh chỉnh (fine-tune) mô hình sẽ được thực hiện ở các bước tiếp theo.

## **2 - KIỂM TRA MÔI TRƯỜNG LOCAL**

Cell này chỉ kiểm tra Python, PyTorch và môi trường local.


In [1]:
import sys
import platform
import torch

print("Python ", sys.version.split()[0])
print("Platform ", platform.platform())
print("PyTorch ", torch.__version__)
print("CUDA available ", torch.cuda.is_available())

Python  3.11.9
Platform  Windows-10-10.0.26200-SP0
PyTorch  2.13.0+cpu
CUDA available  False


## **3 - CONFIG ĐƯỜNG DẪN**

Đường dẫn được tách khỏi source code để không hardcode dữ liệu cá nhân vào loader.

Nếu máy khác, chỉ cần sửa cell này.


In [2]:
from pathlib import Path

PROJECT_ROOT = Path(r"C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep")
DATASET_ROOT = Path(r"C:\Users\Nguyen Ho Vinh  Hien\Downloads\FashionIQ\fashionIQ_dataset")

PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
IMAGES_DIR = DATASET_ROOT / "images"

TRAIN_CSV = PROCESSED_DIR / "fashioniq_triplets_train.csv"
VAL_CSV = PROCESSED_DIR / "fashioniq_triplets_val.csv"
TEST_CSV = PROCESSED_DIR / "fashioniq_triplets_test.csv"

print("PROJECT_ROOT ", PROJECT_ROOT)
print("DATASET_ROOT ", DATASET_ROOT)
print("IMAGES_DIR ", IMAGES_DIR)
print("TRAIN_CSV ", TRAIN_CSV)
print("VAL_CSV ", VAL_CSV)

PROJECT_ROOT  C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep
DATASET_ROOT  C:\Users\Nguyen Ho Vinh  Hien\Downloads\FashionIQ\fashionIQ_dataset
IMAGES_DIR  C:\Users\Nguyen Ho Vinh  Hien\Downloads\FashionIQ\fashionIQ_dataset\images
TRAIN_CSV  C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\data\processed\fashioniq_triplets_train.csv
VAL_CSV  C:\Users\Nguyen Ho Vinh  Hien\khoaluantotnghiep\data\processed\fashioniq_triplets_val.csv


## **4 - KIỂM TRA FILE ĐẦU VÀO**

Cell này xác nhận CSV và thư mục ảnh có tồn tại. Nếu có `False`, dừng lại và sửa đường dẫn trước khi test loader.


In [3]:
checks = {
    "TRAIN_CSV": TRAIN_CSV.exists(),
    "VAL_CSV": VAL_CSV.exists(),
    "TEST_CSV": TEST_CSV.exists(),
    "IMAGES_DIR": IMAGES_DIR.exists(),
}

for name, ok in checks.items():
    print(name, "->", ok)

if not all(checks.values()):
    raise FileNotFoundError("Có đường dẫn chưa đúng")

print("\nTất cả đầu vào đều tồn tại")

TRAIN_CSV -> True
VAL_CSV -> True
TEST_CSV -> True
IMAGES_DIR -> True

Tất cả đầu vào đều tồn tại


## **5 - ĐỌC VÀ KIỂM TRA TRIPLET CSV**

Đây là format FashionIQ mà nhóm đã chuẩn hóa bao gồm
```text
candidate | modifier | target | category
```
- `candidate`: ảnh tham chiếu.
- `modifier`: câu mô tả thay đổi.
- `target`: ảnh đích.
- `category`: dress, shirt hoặc toptee.

Cell này kiểm tra số dòng, tên cột và missing value.


In [4]:
import pandas as pd

train_df = pd.read_csv(TRAIN_CSV, dtype=str)
val_df = pd.read_csv(VAL_CSV, dtype=str)
test_df = pd.read_csv(TEST_CSV, dtype=str)

print("Train rows ", len(train_df))
print("Val rows ", len(val_df))
print("Test rows ", len(test_df))

print("\nColumns ")
print(train_df.columns.tolist())

print("\nTrain missing values ")
print(train_df.isna().sum())

print("\nVal missing values ")
print(val_df.isna().sum())

display(train_df.head())

required_columns = {"candidate", "modifier", "target", "category"}
assert required_columns.issubset(train_df.columns)
assert required_columns.issubset(val_df.columns)

print("\nTriplet CSV PASS")

Train rows  18000
Val rows  6016
Test rows  6118

Columns 
['candidate', 'modifier', 'target', 'category']

Train missing values 
candidate    0
modifier     0
target       0
category     0
dtype: int64

Val missing values 
candidate    0
modifier     0
target       0
category     0
dtype: int64


,candidate,modifier,target,category
0,B003FGW7MK,is solid black with no sleeves and is black wi...,B008BHCT58,dress
1,B008MTHLHQ,is longer and is lighter and longer,B00BZ8GPVO,dress
2,B00EVKYJAC,3/4 sleeve black and white dress and more top ...,B008KZR6WM,dress
3,B003U9VL4W,is patterned and has a halter neckline and is ...,B00684UYEE,dress
4,B00BLY2L2E,is black with no sleeves and is longer,B00EV1B9C2,dress



Triplet CSV PASS


## **6 - KIỂM TRA ÁNH XẠ IMAGE ID SANG FILE .JPG**

Loader gốc UniFashion dùng `.png`, trong khi FashionIQ của nhóm đang dùng `.jpg`. Cell này kiểm tra 5 triplet đầu để xác nhận candidate và target thật sự tồn tại trên ổ đĩa.


In [5]:
sample_rows = train_df.head(5)

for idx, row in sample_rows.iterrows():
    candidate_path = IMAGES_DIR / f"{row['candidate']}.jpg"
    target_path = IMAGES_DIR / f"{row['target']}.jpg"

    print(f"Sample {idx}")
    print("Candidate ", row["candidate"], "->", candidate_path.exists())
    print("Target ", row["target"], "->", target_path.exists())
    print("Modifier ", row["modifier"])
    print("Category ", row["category"])
    print()

Sample 0
Candidate  B003FGW7MK -> True
Target  B008BHCT58 -> True
Modifier  is solid black with no sleeves and is black with straps
Category  dress

Sample 1
Candidate  B008MTHLHQ -> True
Target  B00BZ8GPVO -> True
Modifier  is longer and is lighter and longer
Category  dress

Sample 2
Candidate  B00EVKYJAC -> True
Target  B008KZR6WM -> True
Modifier  3/4 sleeve black and white dress and more top covered and has short sleeve and is black color
Category  dress

Sample 3
Candidate  B003U9VL4W -> True
Target  B00684UYEE -> True
Modifier  is patterned and has a halter neckline and is black with floral patterns
Category  dress

Sample 4
Candidate  B00BLY2L2E -> True
Target  B00EV1B9C2 -> True
Modifier  is black with no sleeves and is longer
Category  dress



## **7 - NẠP LOADER MỚI CHO UNIFASHION**

Loader mới được tách thành file:

```text
src/fashioniq_unifashion_dataset.py
```

Cách này giúp bảo toàn nguyên vẹn mã nguồn gốc của hệ thống UniFashion, dễ dàng theo dõi các tinh chỉnh của nhóm, và đảm bảo không gây xung đột chéo với các tập dữ liệu hay chế độ vận hành khác.

In [6]:
import sys

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.fashioniq_unifashion_dataset import (
    FashionIQTripletDataset,
    unifashion_collate_fn,
)

print("Import loader thành công")

Import loader thành công


## **8 - TẠO PREPROCESS ĐƠN GIẢN CHO SANITY CHECK**

Mặc dù mô hình UniFashion gốc sử dụng kỹ thuật TargetPad và thay đổi kích thước ảnh, trong tuần này nhóm chỉ thiết lập một bước preprocess cơ bản.

Mục đích chính là để kiểm chứng khả năng gộp lô tensor (batching) của DataLoader do hệ thống chưa bước vào giai đoạn train. 

Ở giai đoạn tích hợp chính thức sau này, bước tiền xử lý tạm thời sẽ được thay thế hoàn toàn bằng luồng chuẩn của mã nguồn UniFashion.

In [7]:
from torchvision import transforms

test_preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

print(test_preprocess)

Compose(
    Resize(size=(224, 224), interpolation=bilinear, max_size=None, antialias=True)
    ToTensor()
)


## **9 - KHỞI TẠO TRAIN VÀ VALIDATION DATASET**

Cell này tạo dataset trực tiếp từ file CSV. Nếu thành công nghĩa là loader mới đã bỏ được phụ thuộc vào format JSON riêng của UniFashion.

In [8]:
train_dataset = FashionIQTripletDataset(
    csv_path=TRAIN_CSV,
    images_root=IMAGES_DIR,
    preprocess=test_preprocess,
    strict=True,
)

val_dataset = FashionIQTripletDataset(
    csv_path=VAL_CSV,
    images_root=IMAGES_DIR,
    preprocess=test_preprocess,
    strict=True,
)

print("Train dataset length ", len(train_dataset))
print("Val dataset length ", len(val_dataset))

Train dataset length  18000
Val dataset length  6016


## **10 - SANITY CHECK MỘT SAMPLE**

Đây là cell quan trọng nhất để chứng minh loader đọc đúng dữ liệu.

Cần kiểm tra:
- candidate ID đúng;
- target ID đúng;
- modifier đúng;
- category đúng;
- candidate/target path đúng;
- ảnh đã trở thành tensor.


In [9]:
sample = train_dataset[0]

print("Candidate ID ", sample["candidate_id"])
print("Target ID ", sample["target_id"])
print("Modifier ", sample["modifier"])
print("Category ", sample["category"])

print("\nCandidate path ")
print(sample["candidate_path"])

print("\nTarget path ")
print(sample["target_path"])

print("\nReference tensor shape ", sample["reference_image"].shape)
print("Target tensor shape ", sample["target_image"].shape)

assert sample["reference_image"].shape == (3, 224, 224)
assert sample["target_image"].shape == (3, 224, 224)

print("\nSAMPLE SANITY CHECK PASS")

Candidate ID  B003FGW7MK
Target ID  B008BHCT58
Modifier  is solid black with no sleeves and is black with straps
Category  dress

Candidate path 
C:\Users\Nguyen Ho Vinh  Hien\Downloads\FashionIQ\fashionIQ_dataset\images\B003FGW7MK.jpg

Target path 
C:\Users\Nguyen Ho Vinh  Hien\Downloads\FashionIQ\fashionIQ_dataset\images\B008BHCT58.jpg

Reference tensor shape  torch.Size([3, 224, 224])
Target tensor shape  torch.Size([3, 224, 224])

SAMPLE SANITY CHECK PASS


## **11 - KIỂM TRA NHIỀU SAMPLE**

Cell này kiểm tra 100 sample đầu để kết luận bộ mapping ổn nếu

- candidate tồn tại;
- target tồn tại;
- modifier không rỗng;
- category hợp lệ.

Nếu có lỗi, cell sẽ dừng và chỉ rõ index.


In [10]:
valid_categories = {"dress", "shirt", "toptee"}

for i in range(min(100, len(train_dataset))):
    item = train_dataset[i]

    assert item["candidate_id"]
    assert item["target_id"]
    assert item["modifier"].strip()
    assert item["category"] in valid_categories
    assert Path(item["candidate_path"]).exists()
    assert Path(item["target_path"]).exists()

print("100 SAMPLE CHECK PASS")

100 SAMPLE CHECK PASS


## **12 - TẠO PYTORCH DATALOADER**

Dataset đọc từng sample. PyTorch DataLoader gom nhiều sample thành một batch.

Đây là bước gần nhất với workflow training thật của UniFashion nhưng vẫn chưa chạy model.


In [11]:
from torch.utils.data import DataLoader

train_loader = DataLoader(
    train_dataset,
    batch_size=4,
    shuffle=False,
    num_workers=0,
    collate_fn=unifashion_collate_fn,
)

batch = next(iter(train_loader))

print("Reference batch shape ", batch["reference_images"].shape)
print("Target batch shape ", batch["target_images"].shape)

print("\nModifiers ")
for text in batch["modifiers"]:
    print(" -", text)

print("\nCandidate IDs ", batch["candidate_ids"])
print("Target IDs ", batch["target_ids"])
print("Categories ", batch["categories"])

assert batch["reference_images"].shape == (4, 3, 224, 224)
assert batch["target_images"].shape == (4, 3, 224, 224)

print("\nDATALOADER BATCH PASS")

Reference batch shape  torch.Size([4, 3, 224, 224])
Target batch shape  torch.Size([4, 3, 224, 224])

Modifiers 
 - is solid black with no sleeves and is black with straps
 - is longer and is lighter and longer
 - 3/4 sleeve black and white dress and more top covered and has short sleeve and is black color
 - is patterned and has a halter neckline and is black with floral patterns

Candidate IDs  ['B003FGW7MK', 'B008MTHLHQ', 'B00EVKYJAC', 'B003U9VL4W']
Target IDs  ['B008BHCT58', 'B00BZ8GPVO', 'B008KZR6WM', 'B00684UYEE']
Categories  ['dress', 'dress', 'dress', 'dress']

DATALOADER BATCH PASS


## **13 - KIỂM TRA DATASET THEO CATEGORY**

UniFashion xử lý ba category FashionIQ `dress`, `shirt`, `toptee`.

Loader mới hỗ trợ lọc category trực tiếp để sau này có thể giữ nguyên cách đánh giá từng nhóm của UniFashion.


In [12]:
for category in ["dress", "shirt", "toptee"]:
    ds = FashionIQTripletDataset(
        csv_path=VAL_CSV,
        images_root=IMAGES_DIR,
        preprocess=test_preprocess,
        category=category,
        strict=True,
    )

    first = ds[0]

    print(
        category,
        "| rows =", len(ds),
        "| first candidate =", first["candidate_id"],
        "| first target =", first["target_id"],
    )

dress | rows = 2017 | first candidate = B005X4PL1G | first target = B0084Y8XIU
shirt | rows = 2038 | first candidate = B00CZ7QJUG | first target = B005AD7WZI
toptee | rows = 1961 | first candidate = B008CFZW76 | first target = B008CG1JJ0


## **14 - KIỂM TRA TEST SPLIT**

FashionIQ test có thể không cung cấp target ground-truth vì vậy test split phải dùng `strict=False`.

Cell này chỉ xác nhận loader vẫn đọc được candidate và modifier mà không bắt buộc có target.


In [13]:
test_dataset = FashionIQTripletDataset(
    csv_path=TEST_CSV,
    images_root=IMAGES_DIR,
    preprocess=test_preprocess,
    strict=False,
)

test_sample = test_dataset[0]

print("Test rows ", len(test_dataset))
print("Candidate ", test_sample["candidate_id"])
print("Target ", test_sample["target_id"])
print("Modifier ", test_sample["modifier"])
print("Category ", test_sample["category"])

print("\nTEST SPLIT LOAD PASS")

Test rows  6118
Candidate  B007E66YTO
Target  None
Modifier   yello and more flowing and short and black
Category  dress

TEST SPLIT LOAD PASS


## **15 - KẾT LUẬN**

Việc vượt qua toàn bộ các bước kiểm thử trên khẳng định module nạp dữ liệu (data loader) đã hoàn tất giai đoạn sanity check. Luồng xử lý hiện đã vận hành trơn tru theo đúng tiến trình: nạp dữ liệu từ Triplet CSV, trích xuất các thành phần cốt lõi (ảnh tham chiếu, ảnh đích, câu mô tả), tiền xử lý thành tensor và đóng gói thành từng batch.

Bộ nạp mới này đã khắc phục triệt để hai điểm bất đồng (mismatch) của mã nguồn UniFashion gốc: thay thế định dạng chú thích JSON nội bộ bằng cấu trúc CSV của nhóm, đồng thời linh hoạt nạp ảnh thật định dạng .jpg thay vì giới hạn ở .png. Thành quả này tạo tiền đề vững chắc, sẵn sàng để nhóm tiến hành ghép nối vào kịch bản huấn luyện tinh chỉnh (fine-tuning) ở bước tiếp theo.
